> **Versión corregida (v2.12, ejecución verificada)** — generada a partir de `notebooks/05_politicas_base_capacidad_v2.ipynb` sin sobrescribirlo.
>
> Cambios aplicados:
> - Celda de configuración del repositorio sustituida por el bootstrap portable (`PROJECT_ROOT`/`REPO_ROOT`).
> - Se eliminan `from google.colab import drive`, `drive.mount(...)` y las rutas fijas `/content/drive/MyDrive/PIDA-RL-Diabetes`; resultados en `RESULTS_V2_DIR` (`results/capacidad_v2`, o `results/capacidad_v2_prueba` en modo prueba).
> - Modo prueba: submuestra estratificada por `categoria_riesgo` (sin crear columnas) y capacidad escalada proporcionalmente para conservar la presión de recursos por persona.
>
> Rutas: `PROJECT_ROOT` = datos/modelos/resultados (Drive en Colab, o `PIDA_PROJECT_ROOT`); `REPO_ROOT` = código. Modo prueba: `PIDA_MODO_PRUEBA=1` (cohorte pequeña y pocos episodios).


# 05 — Políticas base y capacidad operativa V2

Este notebook evalúa las políticas **Aleatoria** y **Reglas de riesgo** bajo el mismo entorno, cohorte, horizonte, capacidad y semillas. La salida distingue la acción propuesta por el algoritmo de la acción ejecutada después de aplicar la restricción de recursos.

> El entorno es una simulación metodológica. Sus resultados no constituyen recomendaciones clínicas individuales.

In [ ]:
# === Bootstrap portable: PROJECT_ROOT (corrección v2.12) ===
# PROJECT_ROOT = raíz persistente de datos, modelos y resultados.
# REPO_ROOT    = copia local del código (repositorio GitHub).
# Prioridad: variables de entorno PIDA_PROJECT_ROOT / PIDA_REPO_ROOT,
# después Google Colab + Drive y, por último, ejecución local.
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
import importlib.util
import subprocess

PROJECT_NAME = 'pida-rl-datos-publicos'

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'main')
PROJECT_FOLDER_NAME = 'PIDA-RL-Diabetes'
IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = None
if os.environ.get('PIDA_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['PIDA_PROJECT_ROOT']).expanduser().resolve()
elif IN_COLAB:
    DRIVE_ROOT = next(
        (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
        None,
    )
    if DRIVE_ROOT is None:
        from google.colab import drive
        mount_point = Path('/content/drive')
        if mount_point.exists() and any(mount_point.iterdir()):
            mount_point = Path('/content/gdrive')
        drive.mount(str(mount_point), force_remount=False)
        DRIVE_ROOT = mount_point
    PROJECT_ROOT = DRIVE_ROOT / 'MyDrive' / PROJECT_FOLDER_NAME
else:
    PROJECT_ROOT = (Path.cwd() / PROJECT_FOLDER_NAME).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def _buscar_repo(inicio):
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'src' / 'environment' / 'diabetes_followup_env.py').is_file():
            return candidato
    return None


if os.environ.get('PIDA_REPO_ROOT'):
    REPO_ROOT = Path(os.environ['PIDA_REPO_ROOT']).expanduser().resolve()
else:
    REPO_ROOT = _buscar_repo(Path.cwd().resolve()) or (
        Path('/content/pida-rl-datos-publicos') if IN_COLAB
        else Path.cwd().resolve() / 'pida-rl-datos-publicos'
    )
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
elif not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(f'REPO_ROOT no contiene la carpeta src: {REPO_ROOT}')
elif IN_COLAB and (REPO_ROOT / '.git').is_dir():
    # En Colab se alinea la copia temporal con la rama de trabajo (sin push ni commit).
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', '--ff-only', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=False)

SRC_ROOT = REPO_ROOT / 'src'
for ruta in (str(REPO_ROOT), str(SRC_ROOT)):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)
os.chdir(REPO_ROOT)

# Estructura persistente común (antes: rutas fijas /content/drive/MyDrive/PIDA-RL-Diabetes).
PERSISTENT_ROOT = PROJECT_ROOT  # alias de compatibilidad con los notebooks originales
MYDRIVE_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

# Modo prueba: cohorte pequeña y pocos episodios (PIDA_MODO_PRUEBA=1).
MODO_PRUEBA = os.environ.get('PIDA_MODO_PRUEBA', '0') == '1'
N_COHORTE_PRUEBA = int(os.environ.get('PIDA_N_COHORTE', '120'))
SEMILLA_MUESTRA_PRUEBA = 2026
RESULTS_V2_DIR = RESULTS_DIR / ('capacidad_v2_prueba' if MODO_PRUEBA else 'capacidad_v2')


def muestra_cohorte_prueba(cohorte, n=None, semilla=SEMILLA_MUESTRA_PRUEBA):
    """Devuelve (cohorte, factor). En modo prueba toma una submuestra
    estratificada por categoria_riesgo (bajo/medio/alto); no crea columnas."""
    n = N_COHORTE_PRUEBA if n is None else int(n)
    if not MODO_PRUEBA or len(cohorte) <= n:
        return cohorte, 1.0
    fraccion = n / len(cohorte)
    muestra = (
        cohorte.groupby('categoria_riesgo', group_keys=False)
        .sample(frac=fraccion, random_state=semilla)
        .sort_values('FOLIO_INT')
        .reset_index(drop=True)
    )
    return muestra, len(muestra) / len(cohorte)


def escalar_capacidad(capacidad, factor):
    """Mantiene la presión de capacidad (recursos/persona) al reducir la cohorte."""
    return capacidad if factor >= 1.0 else max(1, int(round(capacidad * factor)))

print('Entorno:', 'Google Colab' if IN_COLAB else 'local')
print('PROJECT_ROOT (datos/resultados):', PROJECT_ROOT)
print('REPO_ROOT (código):', REPO_ROOT)
print('Modo prueba:', MODO_PRUEBA)

for _mod, _pip in [('gymnasium', 'gymnasium'), ('pandas', 'pandas')]:
    if importlib.util.find_spec(_mod) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', _pip])
try:
    print('Commit evaluado:', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip())
except Exception as exc:  # copia sin .git
    print('Commit evaluado: no disponible', exc)


## Configuración del experimento

Modifica `CAPACIDAD_MENSUAL` para representar los recursos disponibles en cada mes. Los costos son supuestos operativos explícitos y ajustables, no costos monetarios ni clínicos observados.

In [ ]:
CAPACIDAD_MENSUAL = 100
COSTOS_ACCION = {0: 0, 1: 1, 2: 3, 3: 5}
HORIZONTE_MESES = 12
SEMILLAS_EVALUACION = (42, 123, 2026)

NOMBRES_ACCION = {
    0: 'Sin contacto',
    1: 'Mensaje',
    2: 'Llamada',
    3: 'Teleorientación',
}

assert CAPACIDAD_MENSUAL >= 0
assert set(COSTOS_ACCION) == {0, 1, 2, 3}
print('Capacidad mensual:', CAPACIDAD_MENSUAL)
print('Costos:', COSTOS_ACCION)

## Cohorte de evaluación

Se carga la cohorte preparada por el notebook 03 desde Google Drive. No se generan pacientes nuevos en este notebook.

In [ ]:
import numpy as np
import pandas as pd

COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'
OUTPUT_DIR = RESULTS_V2_DIR / 'politicas_base'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not COHORT_PATH.is_file():
    raise FileNotFoundError(f'No existe {COHORT_PATH}. Ejecuta primero el notebook 03.')

cohorte = pd.read_csv(COHORT_PATH, dtype={'FOLIO_INT': 'string'})
columnas_requeridas = [
    'FOLIO_INT', 'edad', 'sexo', 'anios_con_diabetes',
    'insulina_diaria', 'depresion', 'consultas_control_12m',
    'num_hospitalizaciones', 'num_complicaciones',
    'score_riesgo', 'categoria_riesgo',
]
faltantes = [c for c in columnas_requeridas if c not in cohorte.columns]
if faltantes:
    raise KeyError('Faltan columnas: ' + ', '.join(faltantes))
if cohorte['FOLIO_INT'].duplicated().any():
    raise ValueError('FOLIO_INT contiene identificadores duplicados.')
if not cohorte['categoria_riesgo'].isin(['bajo', 'medio', 'alto']).all():
    raise ValueError('Se encontraron categorías de riesgo no válidas.')

# Modo prueba: cohorte pequeña estratificada y capacidad proporcional.
N_COHORTE_COMPLETA = len(cohorte)
cohorte, FACTOR_COHORTE = muestra_cohorte_prueba(cohorte)
CAPACIDAD_MENSUAL = escalar_capacidad(CAPACIDAD_MENSUAL, FACTOR_COHORTE)
assert set(cohorte['categoria_riesgo'].unique()) <= {'bajo', 'medio', 'alto'}
print(f'Cohorte usada: {len(cohorte)} de {N_COHORTE_COMPLETA} perfiles | capacidad mensual: {CAPACIDAD_MENSUAL}')
print(cohorte['categoria_riesgo'].value_counts().to_dict())

print('Cohorte:', cohorte.shape)
print('Ruta de resultados:', OUTPUT_DIR)
display(cohorte.head())

## Evaluación común

Ambas políticas se ejecutan con las mismas semillas. Para cada política se crea una instancia nueva del entorno con idéntica configuración.

In [ ]:
from src.environment.diabetes_followup_env import DiabetesFollowUpEnv
from src.evaluation.evaluate_policies import evaluar_politicas
from src.evaluation.reporting import construir_reporte, exportar_reporte_csv
from src.policies.baselines import POLITICAS_BASE

def crear_entorno():
    return DiabetesFollowUpEnv(
        cohorte_df=cohorte,
        capacidad_mensual=CAPACIDAD_MENSUAL,
        costos_accion=COSTOS_ACCION,
        horizonte=HORIZONTE_MESES,
    )

resultados = evaluar_politicas(
    env_factory=crear_entorno,
    politicas=POLITICAS_BASE,
    semillas=SEMILLAS_EVALUACION,
)
reporte = construir_reporte(resultados)
rutas = exportar_reporte_csv(reporte, OUTPUT_DIR)
print('Archivos exportados:')
for nombre, ruta in rutas.items():
    print(f'- {nombre}: {ruta}')

## Validaciones de capacidad

Estas comprobaciones detienen el notebook si una política supera la capacidad o si el registro queda incompleto.

In [ ]:
detalle = reporte['detalle']
assert (detalle['recursos_despues'] >= 0).all(), 'Hay recursos negativos.'
assert (detalle['recursos_usados_mes'] <= detalle['capacidad_mensual']).all(), 'Se excedió la capacidad.'
assert detalle['accion_propuesta'].isin([0, 1, 2, 3]).all()
assert detalle['accion_ejecutada'].isin([0, 1, 2, 3]).all()
filas_esperadas = len(cohorte) * HORIZONTE_MESES * len(SEMILLAS_EVALUACION) * len(POLITICAS_BASE)
assert len(detalle) == filas_esperadas, f'Se esperaban {filas_esperadas} decisiones y se obtuvieron {len(detalle)}.'
print('Validaciones correctas:', len(detalle), 'decisiones registradas.')

## Resultados para revisión

La primera tabla compara métricas por algoritmo. Las tablas siguientes muestran las clasificaciones propuestas y ejecutadas; su diferencia representa ajustes causados por la capacidad disponible.

In [ ]:
display(reporte['resumen_algoritmos'].round(4))
display(reporte['clasificaciones_propuestas'].round(2))
display(reporte['clasificaciones_ejecutadas'].round(2))
display(detalle.head(20))

## Interpretación

- **Acción propuesta:** clasificación emitida por la política.
- **Acción ejecutada:** clasificación permitida después de descontar recursos.
- **Ajuste de capacidad:** la propuesta no cabía y se redujo a la intervención viable de mayor intensidad.
- **Cobertura de alto riesgo:** proporción de decisiones de riesgo alto con alguna acción ejecutada.
- **Utilización de capacidad:** recursos consumidos divididos entre recursos disponibles durante el episodio.

Los resultados describen comportamiento dentro del simulador y no efectividad clínica observada.